# 02 - Primary preprocessing
Runs the leakage-safe pipeline in `src/preprocess.py` and shows each decision's evidence. **No models are trained.**

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 80)
from src.utils import load_config, reports_dir, outputs_dir
cfg = load_config()          # uses $DEPRESSION_CONFIG if set, else configs/config.yaml
print("config:", cfg["_config_path"])

config: /tmp/config_run.yaml


In [2]:
from src.preprocess import run_pipeline, validate_outputs
res = run_pipeline(cfg, verbose=False)
res['clean_log']

,step,n_rows_affected
0,"weight below adult minimum -> weight, bmi set ...",20
1,weekday: asleep==0 & in-bed==0 -> device non-w...,1206
2,weekend: asleep==0 & in-bed==0 -> device non-w...,2041
3,"steps_awake_mean < 500 (flag only, kept)",20
4,"bmi > 60 (flag only, kept)",178


### Participant-wise splits (no participant appears twice)

In [3]:
res['split_summary']

,split,rows,participants,positive_rate_ge10,mean_phq9_end,mean_phq9_start
0,train,7610,2835,0.316820,7.497503,7.707884
1,val,1630,593,0.317178,7.384049,7.412270
2,test,1626,608,0.317343,7.411439,7.696802


### Features: what was excluded and why

In [4]:
import json
out = outputs_dir(cfg, 'primary')
fs = json.load(open(out / 'feature_sets.json'))
print('main feature columns:', len(fs['main_feature_columns']))
print('timing-uncertain extras (separate file):', len(fs['timing_uncertain_extra_columns']))
print('excluded outcome-period columns:', fs['excluded_outcome_columns'])
print('sensitive (fairness only):', fs['sensitive_fairness_only'])
print('dropped near-duplicates (dropped -> kept):', fs['dropped_correlated(dropped->kept)'])
print('missing indicators:', len(fs['missing_indicator_columns']))

main feature columns: 139
timing-uncertain extras (separate file): 20
excluded outcome-period columns: ['phq9_cat_end', 'phq9_cat_start', 'phq9_score_end']
sensitive (fairness only): ['race_asian', 'race_black', 'race_hispanic', 'race_other', 'race_white', 'sex']
dropped near-duplicates (dropped -> kept): {'sleep_ratio_asleep_in_bed__intercept': 'sleep__main_efficiency__intercept', 'sleep_ratio_asleep_in_bed__intercept_': 'sleep__main_efficiency__intercept_', 'sleep_ratio_asleep_in_bed_mean_recent': 'sleep_ratio_asleep_in_bed_weekday_mean'}
missing indicators: 18


### Independent validation of the written files

In [5]:
validate_outputs(cfg)

,check,passed,detail
0,no participant in two splits,True,train=2835 val=593 test=608
1,train: X rows == y rows == meta rows,True,7610
2,train: no NaN/inf in X,True,
3,train: no NaN/inf in timing-uncertain extra,True,
4,train: columns identical to final_feature_list,True,
5,train: targets complete,True,
6,val: X rows == y rows == meta rows,True,1630
7,val: no NaN/inf in X,True,
8,val: no NaN/inf in timing-uncertain extra,True,
9,val: columns identical to final_feature_list,True,


### Personal-baseline / history features (experimental)
Only strictly earlier periods of the same participant are used. No daily data exists, so a 2-4 week daily baseline is **not** reconstructed.

In [6]:
from src.feature_engineering import history_coverage
hist = pd.read_parquet(outputs_dir(cfg, 'temporal') / 'temporal_features_experimental.parquet')
c = pd.read_parquet(out / 'cleaned_primary_dataset.parquet')
lab = c.loc[c.is_labelled, ['participant_id', 'period']]
history_coverage(hist, lab, cfg['temporal']['min_history'])

{'labelled_rows': 10866,
 'rows_with_>=2_prior_periods': 9841,
 'share': 0.9056690594515001}

In [7]:
import pytest, subprocess
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT / 'tests' / 'test_primary.py')], cwd=ROOT)

...                                                                      [100%]
3 passed in 1.36s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '/home/claude/depression-preprocessing/tests/test_primary.py'], returncode=0)